In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib


In [ ]:
df = pd.read_parquet("favorita_model_ready_2013_2015.parquet")


## Inference-ready Features

In [ ]:
FEATURES_KAGGLE = [
    # identifiers
    "store_nbr", "item_nbr",

    # categorical structure
    "family", "class", "city", "cluster",

    # promotions & item properties
    "onpromotion", "perishable",

    # calendar
    "month", "dayofweek", "weekofyear", "is_weekend",

    # external (known / forward-filled)
    "dcoilwtico", "is_holiday",

    # time-series
    "lag_7", "lag_14", "lag_28",
    "rolling_7", "rolling_14"
]


In [ ]:
CATEGORICAL_FEATURES = [
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "city",
    "cluster"
]


## Apply categorical to features

In [ ]:
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype("category")


In [ ]:
category_maps_kaggle = {
    col: df[col].cat.categories
    for col in CATEGORICAL_FEATURES
}

joblib.dump(category_maps_kaggle, "category_maps_kaggle.pkl")


['category_maps_kaggle.pkl']

## Time-based split

In [ ]:
train_end = "2014-12-31"
valid_end = "2015-06-30"

train_df = df[df["date"] <= train_end]
valid_df = df[(df["date"] > train_end) & (df["date"] <= valid_end)]
test_df  = df[df["date"] > valid_end]

X_train = train_df[FEATURES_KAGGLE]
X_valid = valid_df[FEATURES_KAGGLE]
X_test  = test_df[FEATURES_KAGGLE]


## Prepare target

In [ ]:
y_train_raw = np.clip(train_df["unit_sales"].values, 0, None)
y_valid_raw = np.clip(valid_df["unit_sales"].values, 0, None)
y_test_raw  = np.clip(test_df["unit_sales"].values, 0, None)

y_train = np.log1p(y_train_raw)
y_valid = np.log1p(y_valid_raw)
y_test  = np.log1p(y_test_raw)


## Train P90 Quantile Model

In [ ]:
p90_V2 = lgb.LGBMRegressor(
    objective="quantile",
    alpha=0.90,
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=64,
    min_data_in_leaf=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

p90_V2.fit(
    X_train,
    y_train,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(X_valid, y_valid)],
    eval_metric="quantile",
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(100)
    ]
)


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.726557 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2522
[LightGBM] [Info] Number of data points in the train set: 8516559, number of used features: 19
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] Start training from score 3.737670
Training until validation scores don't improve for 50

LGBMRegressor(alpha=0.9, colsample_bytree=0.8, learning_rate=0.03,
              min_data_in_leaf=200, n_estimators=3000, n_jobs=-1, num_leaves=64,
              objective='quantile', random_state=42, subsample=0.8)

## Sanity-check decision behavior

In [ ]:
pred_log = p90_V2.predict(X_test, num_iteration=p90_V2.best_iteration_)
order_qty = np.expm1(pred_log).clip(0)

actual = y_test_raw

service_level = np.mean(actual <= order_qty)
overage = np.maximum(order_qty - actual, 0)
underage = np.maximum(actual - order_qty, 0)

service_level, overage.mean(), underage.mean()


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200


(np.float64(0.8927426912476398),
 np.float64(11.239515628877534),
 np.float64(0.9434999619566725))

In [ ]:
from sklearn.metrics import mean_squared_log_error

# RMSLE diagnostic (note: q90_V2 does NOT optimize RMSLE)
rmsle_p90_V2 = np.sqrt(
    mean_squared_log_error(
        y_test_raw,                 # actual (clipped nonnegative)
        np.clip(order_qty, 0, None) # predicted order quantity (nonnegative)
    )
)

print("Service level:", float(service_level))
print("Avg overage:", float(overage.mean()))
print("Avg underage:", float(underage.mean()))
print("RMSLE (diagnostic):", float(rmsle_p90_V2))


Service level: 0.8927426912476398
Avg overage: 11.239515628877534
Avg underage: 0.9434999619566725
RMSLE (diagnostic): 0.7749290154038626


In [ ]:
p90_V2.booster_.save_model("favorita_lgb_q90_V2.txt")


## P95 model definition

In [ ]:
import lightgbm as lgb

params_p95 = {
    "objective": "quantile",
    "alpha": 0.95,          # 👈 ONLY CHANGE
    "metric": "quantile",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": 42,
}


## Train P95 with early stopping

In [ ]:
model_p95 = lgb.LGBMRegressor(**params_p95, n_estimators=3000)

model_p95.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=CATEGORICAL_FEATURES,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(100),
    ],
)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's quantile: 0.0463148
[200]	valid_0's quantile: 0.0457225
[300]	valid_0's quantile: 0.0456635
[400]	valid_0's quantile: 0.0458101
Early stopping, best iteration is:
[303]	valid_0's quantile: 0.0456568


LGBMRegressor(alpha=0.95, bagging_fraction=0.8, bagging_freq=1,
              feature_fraction=0.8, learning_rate=0.05, metric='quantile',
              min_data_in_leaf=100, n_estimators=3000, num_leaves=64,
              objective='quantile', seed=42, verbosity=-1)

In [ ]:
y_pred_log = model_p95.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)


In [ ]:
service_level = np.mean(y_true <= y_pred)
overage = np.mean(np.maximum(y_pred - y_true, 0))
underage = np.mean(np.maximum(y_true - y_pred, 0))

service_level, overage, underage

(np.float64(0.9387913804051619),
 np.float64(14.728574244237208),
 np.float64(0.6208348656983412))

In [ ]:
from sklearn.metrics import mean_squared_log_error

rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred))
rmsle

np.float64(0.881097536071878)

In [ ]:
model_p95.booster_.save_model("favorita_lgb_p95.txt")
